In [1]:
!pip install langchain-groq langchain dotenv langchain-community langchain-text-splitters langchain-huggingface -qU

In [2]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Anthropic")
docs = loader.load()

/var/folders/8x/84jtjkf95mv22ccykhl3lz6h0000gq/T/ipykernel_59449/3054827713.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100, 
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(docs)
print(f"Number of chunks created: {len(chunks)}")


Number of chunks created: 104


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14004.13it/s]


In [32]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
model="openai/gpt-oss-120b" # "openai/gpt-oss-20b" "qwen/qwen3.8-27b", "openai/gpt-oss-120b"


llm = chat_groq = ChatGroq(
    model=model,
    api_key=GROQ_API_KEY,
    max_tokens=1000,
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer clearly and concisely using the provided context. If the context does not contain the answer, say so.\n\nContext:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = prompt_template | llm

In [38]:
from langchain_core.messages import AIMessage, HumanMessage

def add_messages(messages, question, answer):
    messages.extend([
        HumanMessage(content=question),
        AIMessage(content=answer),
    ])

In [39]:
chat_history = []
print("Chat started. Type 'exit' or 'quit' to stop.")

while True:
    question = input("You: ").strip()
    if question.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break
    if not question:
        continue

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer = chain.invoke({
        "question": question,
        "context": context,
        "history": chat_history,
    })
    print(f"Assistant: {answer.content}\n")
    add_messages(chat_history, question, answer.content)

Chat started. Type 'exit' or 'quit' to stop.
Assistant: Hello! How can I assist you today?

Assistant: The CEO of Anthropic is **Dario Amodei**.

Chat ended.
